In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from datetime import date

# Initialize Spark session (skip this line if you are already in a Databricks notebook)
spark = SparkSession.builder.appName("RetentionCohortAnalysis").getOrCreate()

# 1. Define the schema based on the DDL
schema = StructType([
    StructField("action_id", IntegerType(), False),
    StructField("user_id", IntegerType(), True),
    StructField("action_date", DateType(), True),
    StructField("action_type", StringType(), True),
    StructField("platform", StringType(), True)
])

# 2. Replicate the line-by-line inserts
data = [
    (1, 101, date(2024, 1, 1), 'signup', 'iOS'),
    (2, 101, date(2024, 1, 15), 'login', 'iOS'),
    (3, 102, date(2024, 1, 2), 'signup', 'Web'),
    (4, 103, date(2024, 2, 1), 'signup', 'Android')
]

# 3. Create the DataFrame
df = spark.createDataFrame(data, schema=schema)

# 4. Create the temporary view for SQL practice
df.createOrReplaceTempView("user_actions")

# Verify the setup
spark.sql("SELECT * FROM user_actions").show()

In [0]:
spark.sql("""
    WITH first_actions as (
    select 
        user_id,
        DATE_TRUNC('WEEK', MIN(action_date)) AS cohort_week,
        MIN(action_date) as min_signup_date
    FROM user_actions
    group by user_id
    )
    select f.cohort_week, 
    COUNT(DISTINCT f.user_id) AS cohort_size,
    COUNT(DISTINCT CASE WHEN 
        u.action_date > f.min_signup_date AND
        u.action_date <= f.cohort_week + INTERVAL '30 days'
    THEN f.user_id 
    END) AS retained_user_30d
    from first_actions as f join user_actions as u
    on f.user_id = u.user_id
    group by f.cohort_week

""").display()